# 2D Design Template

# Overview

The purpose of this project is for you to apply what you have learnt in this course. This includes working with data and visualizing it, create model of linear regression, as well as using metrics to measure the accuracy of your model. 

Please find the project handout description in the following [link](https://edimension.sutd.edu.sg/webapps/blackboard/content/listContent.jsp?course_id=_5261_1&content_id=_184406_1).


## Deliverables

You need to submit this Jupyter notebook together with the dataset into Vocareum. Use the template in this notebook to work on this project. You are free to edit or add more cells if needed

## Students Submission

Student's Name:
- Name 1
- Name 2
- ...

### Problem Statement

Climatic factors include rainfall and temperature. These abiotic components, together with pesticides and soil, are environmental factors that influence plant growth and development

Indonesia's agricultural sector, which forms the primary livelihood for most of the population [1], largely relies on traditional planting practices to determine the types of crop to be planted. These practices, passed down through generations, are increasingly challenged by shifting climate conditions. Changes in climate, such as variations in temperature and rainfall, have negatively impact on the productivity of agricultural land, particularly for food crops [2][3]. Consequently, the traditional methods used by farmers are no longer as effective in ensuring optimal crop yields.

To address this challenge, improving farmers' productivity through the adoption of more precise and data-driven methods for crop selection is critical. Climatic factors, including rainfall, temperature, sunshine period and wind speed, play an important role in plant growth and development. Minimizing human error in crop selection by leveraging these factors can enhance agricultural efficiency and food security.


**How might we assist farmers in determining what to plant at the start of the season?**

### Dataset

- Describe your dataset.
- Put the link to the sources of your raw dataset.
- Put python codes for loading the data into pandas dataframe(s). The data should be the raw data downloaded from the source. No pre-processing using any software (excel, python, etc) yet. Include this dataset in your submission
- Explain each column of your dataset (can use comment or markdown)
- State which column is the dependent variable (target) and explain how it is related to your problem statement
- State which columns are the independent variables (features) and describe your hypothesis on why these features can predict the target variable

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.axes as axes
import seaborn as sns
import missingno as msno


In [23]:

weatherdf: pd.DataFrame = pd.read_csv("D:/term3_dtp/weather/climate_data.csv") # provide station number
provincedf: pd.DataFrame = pd.read_csv("D:/term3_dtp/weather/province_detail.csv") # provide province number
stationdf: pd.DataFrame = pd.read_csv("D:/term3_dtp/weather/station_detail.csv") # provide province number that the station belong to

corn_proddf: pd.DataFrame = pd.read_excel("D:/term3_dtp/crop_prod/BDSP_EXP_251124_1347_2431.xlsx", engine='openpyxl', sheet_name="Jagung") # Corn
rice_proddf: pd.DataFrame = pd.read_excel("D:/term3_dtp/crop_prod/BDSP_EXP_251124_1347_2431.xlsx", engine='openpyxl', sheet_name="Padi") # Rice
mungbean_proddf: pd.DataFrame = pd.read_excel("D:/term3_dtp/crop_prod/BDSP_EXP_251124_1347_2431.xlsx", engine='openpyxl', sheet_name="Kacang Hijau") # Mung Beans
peanut_proddf: pd.DataFrame = pd.read_excel("D:/term3_dtp/crop_prod/BDSP_EXP_251124_1347_2431.xlsx", engine='openpyxl', sheet_name="Kacang Tanah") # Peanuts
sago_proddf: pd.DataFrame = pd.read_excel("D:/term3_dtp/crop_prod/BDSP_EXP_251124_1347_2431.xlsx", engine='openpyxl', sheet_name="Sagu") # Sago


# Drop the last column for each DataFrame since it is irrelevant 
corn_proddf = corn_proddf.iloc[:, :-1]
rice_proddf = rice_proddf.iloc[:, :-1]
mungbean_proddf = mungbean_proddf.iloc[:, :-1]
peanut_proddf = peanut_proddf.iloc[:, :-1]
sago_proddf = sago_proddf.iloc[:, :-1]

# Assign crop names to each DataFrame
corn_proddf["Crop"] = "Corn"
rice_proddf["Crop"] = "Rice"
mungbean_proddf["Crop"] = "Mung Beans"
peanut_proddf["Crop"] = "Peanuts"
sago_proddf["Crop"] = "Sago"

# Combine all DataFrames
crop_prod = pd.concat([corn_proddf, rice_proddf, mungbean_proddf, peanut_proddf, sago_proddf], ignore_index=True)

# Reshape the data
crop_proddf = crop_prod.melt(
    id_vars=["No", "Provinsi", "Crop"],
    var_name="Year",
    value_name="Production"
)

display(weatherdf.head())
display(crop_proddf)

,date,Tn,Tx,Tavg,RH_avg,RR,ss,ff_x,ddd_x,ff_avg,ddd_car,station_id
0,01-01-2010,21.4,30.2,27.1,82.0,9.0,0.5,7.0,90.0,5.0,E,96001
1,02-01-2010,21.0,29.6,25.7,95.0,24.0,0.2,6.0,90.0,4.0,E,96001
2,03-01-2010,20.2,26.8,24.5,98.0,63.0,0.0,5.0,90.0,4.0,E,96001
3,04-01-2010,21.0,29.2,25.8,90.0,0.0,0.1,4.0,225.0,3.0,SW,96001
4,05-01-2010,21.2,30.0,26.7,90.0,2.0,0.4,NaN,NaN,NaN,NaN,96001


,No,Provinsi,Crop,Year,Production
0,1,Aceh,Corn,2013,0.0
1,2,Sumatera Utara,Corn,2013,1182925.0
2,3,Sumatera Barat,Corn,2013,547437.0
3,4,Riau,Corn,2013,28052.0
4,5,Jambi,Corn,2013,25489.0
...,...,...,...,...,...
1920,31,Maluku,Sago,2023,NaN
1921,32,Maluku Utara,Sago,2023,NaN
1922,33,Papua Barat,Sago,2023,NaN
1923,34,Papua,Sago,2023,NaN


### Clean & Analyze data
Use python code to:
- Clean your data
- Calculate Descriptive Statistics and other statistical analysis
- Visualization with meaningful analysis description

In [25]:
weatherdf.info() #check datatype

weatherdf.isnull().sum() #check null


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 589265 entries, 0 to 589264
Data columns (total 12 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   date        589265 non-null  object 
 1   Tn          565882 non-null  float64
 2   Tx          551529 non-null  float64
 3   Tavg        544160 non-null  float64
 4   RH_avg      541083 non-null  float64
 5   RR          463881 non-null  float64
 6   ss          545544 non-null  float64
 7   ff_x        579051 non-null  float64
 8   ddd_x       576137 non-null  float64
 9   ff_avg      579138 non-null  float64
 10  ddd_car     575526 non-null  object 
 11  station_id  589265 non-null  int64  
dtypes: float64(9), int64(1), object(2)
memory usage: 53.9+ MB


date               0
Tn             23383
Tx             37736
Tavg           45105
RH_avg         48182
RR            125384
ss             43721
ff_x           10214
ddd_x          13128
ff_avg         10127
ddd_car        13739
station_id         0
dtype: int64

### Handle Missing Values

To handle missing values in the weather dataset, fills missing values using the monthly averages.

For **RR**, it is likely that the NaN datas are actually days that it did not rain



In [ ]:
weatherdf['date'] = pd.to_datetime(weatherdf['date'], format='%d-%m-%Y')

weatherdf['month'] = (weatherdf.index // 30) % 12 + 1

# Fill missing values with monthly averages
weatherdf['Tn'] = weatherdf.groupby('month')['Tn'].transform(lambda x: x.fillna(x.mean()))
weatherdf['Tx'] = weatherdf.groupby('month')['Tx'].transform(lambda x: x.fillna(x.mean()))
weatherdf['Tavg'] = weatherdf.groupby('month')['Tavg'].transform(lambda x: x.fillna(x.mean()))
weatherdf['RH_avg'] = weatherdf.groupby('month')['RH_avg'].transform(lambda x: x.fillna(x.mean()))
weatherdf['ss'] = weatherdf.groupby('month')['ss'].transform(lambda x: x.fillna(x.mean()))
weatherdf['ff_x'] = weatherdf.groupby('month')['ff_x'].transform(lambda x: x.fillna(x.mean()))
weatherdf['ddd_x'] = weatherdf.groupby('month')['ddd_x'].transform(lambda x: x.fillna(x.mean()))
weatherdf['ff_avg'] = weatherdf.groupby('month')['ff_avg'].transform(lambda x: x.fillna(x.mean()))
weatherdf['ddd_car'] = weatherdf.groupby('month')['ddd_car'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan))

# Fill in missing values for rainfall to be 0 as it is likely that it did not rain 
weatherdf['RR'] = weatherdf['RR'].fillna(0)

# Check for remaining missing values
weatherdf.isnull().sum()

date          0
Tn            0
Tx            0
Tavg          0
RH_avg        0
RR            0
ss            0
ff_x          0
ddd_x         0
ff_avg        0
ddd_car       0
station_id    0
month         0
dtype: int64

In [17]:
# descriptive statistics
weatherdf.describe()

,date,Tn,Tx,Tavg,RH_avg,RR,ss,ff_x,ddd_x,ff_avg,station_id,month
count,589265,589265.000000,589265.000000,589265.000000,589265.000000,589265.000000,589265.000000,589265.000000,589265.000000,589265.000000,589265.000000,589265.000000
mean,2015-08-09 23:46:38.410222592,23.312152,31.528961,26.855544,82.488660,6.833665,5.083250,4.709594,188.483616,1.956683,96832.949230,6.499529
min,2010-01-01 00:00:00,0.000000,0.000000,0.000000,24.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,96001.000000,1.000000
25%,2012-08-27 00:00:00,23.000000,30.600000,26.300000,79.000000,0.000000,2.800000,3.000000,100.000000,1.000000,96293.000000,3.000000
50%,2015-10-23 00:00:00,24.000000,31.700000,27.000000,83.000000,0.000000,5.159305,4.000000,183.623235,2.000000,96797.000000,6.000000
75%,2018-06-09 00:00:00,24.900000,32.800000,27.900000,87.000000,5.800000,7.500000,6.000000,270.000000,3.000000,97240.000000,9.000000
max,2020-12-31 00:00:00,246.000000,334.000000,141.600000,7520.000000,1965.500000,705.000000,185.000000,931.000000,160.000000,97980.000000,12.000000
std,NaN,2.234987,2.236483,1.863986,13.739545,16.299274,3.138437,2.589551,106.453131,1.787798,542.419161,3.451869


### Descriptive Statistics Analysis

**Tn**, **Tx** and **Tavg** contain minimum and maximum temperature values that are clearly incorrect, as temperatures in Indonesia do not drop to 0°C, and the recorded maximum temperatures exceed realistic levels.

**RH_avg** with the maximum value far beyond the possible range for humidity (humidity should be within 0% to 100%).

**RR** minimum value is -1 mm, which is not possible for rainfall. Rainfall values must be non-negative. Its' maximum value is 1965.5 mm. While rare, this could indicate an extreme weather event, but we cross-checked with historical data and found no such event.

**ss** maximum value is 705 hours, which is unrealistic as it exceeds the total hours in a month.

**ff_x** maximum value is 185 m/s, which is extremely high and would classify as a catastrophic wind event (e.g., a super typhoon). We cross-checked with historical data and find no such event, the highest wind speeds recored in Indonesia only reach up to 250 km/h or 70 m/s in a typhoon.

**ddd_x** maximum value is 931°. Wind direction is measured in degrees and should range from 0° to 360°

**ff_avg** maximum value is 160 m/s, which is unrealistically high and exceeds the highest wind speed recorded in Indonesia.

These anomalies are likely data entry errors and will be addressed through outlier detection methods.


### Handle outlier

In [5]:
# visualization with analysis

### Features and Target Preparation

Prepare features and target for model training.

In [0]:
# put Python code to prepare your features and target

### Building Model

Use python code to build your model. Give explanation on this process.

In [0]:
# put Python code to build your model

### Evaluating the Model

- Describe the metrics of your choice
- Evaluate your model performance

In [6]:
# put Python code to test & evaluate the model

### Improving the Model

- Improve the models by performing any data processing techniques or hyperparameter tuning.
- You can repeat the steps above to show the improvement as compared to the previous performance

Note:
- You should not change or add dataset at this step
- You are allowed to use library such as sklearn for data processing (NOT for building model)
- Make sure to have the same test dataset so the results are comparable with the previous model 
- If you perform hyperparameter tuning, it will require you to split your training data further into train and validation dataset

In [0]:
# Re-iterate the steps above with improvement

### Discussion and Analysis

- Analyze the results of your metrics.
- Explain how does your analysis and machine learning help to solve your problem statement.
- Conclusion

### References

[1] _Statista. (2024, March 15). Number of workers in Indonesia 2023, by sector. https://www.statista.com/statistics/994498/employment-numbers-by-industry-indonesia/_

[2] _Zhao, C. (2017). Temperature increase reduces global yields of major crops in four independent estimates. Proceedings of the National Academy of Sciences, 114(35), 9326–9331. https://doi.org/10.1073/pnas.1701762114_

[3] _Springmann, M. (2016). Global and regional health effects of future food production under climate change: a modelling study. The Lancet, 387(10031), 1937–1946. https://doi.org/10.1016/s0140-6736(15)01156-3_

[4] _Tebaldi, C., & Lobell, D. B. (2008). Towards probabilistic projections of climate change impacts on global crop yields. Geophysical Research Letters, 35(8). https://doi.org/10.1029/2008gl033423_
